# Qdrant RAG

In [19]:
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown, Latex


load_dotenv("../.env")


True

In [42]:
client = QdrantClient("http://localhost:6333")
COLLECTION_NAME = "llm-zoomcamp-rag"

class QdrantRAG:

    def __init__(self, llm, ss_embedding_model: str, template: str) -> None:
        self.model = llm
        self.embedding_model = ss_embedding_model
        self.client = QdrantClient("http://localhost:6333")
        self.prompt = template

    def search(self, query: str, limit: int  = 5):
        
        results = self.client.query_points(
        collection_name=COLLECTION_NAME,
        query=models.Document( #embed the query text locally with "jinaai/jina-embeddings-v2-small-en"
            text=query,
            model=self.embedding_model 
            ),
            limit=limit, # top closest matches
            with_payload=True #to get metadata in the results
        )
        
        return "\n".join([i.payload["text"] for i in results.points])

    def query(self, question: str):
        search_results = self.search(question)
        chain = self.prompt | self.model
        response = chain.invoke(
            {
                "context": search_results, # elasticsearch 
                "question": question,
            }
        )
        # print(search_results)
        return question, search_results, response
        



In [45]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

rag = QdrantRAG(
    llm=llm, 
    ss_embedding_model="jinaai/jina-embeddings-v2-small-en",
    template=ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
                    Use only the facts from the CONTEXT when answering the QUESTION.
                    CONTEXT: 
                    {context}
                    """,
                ),
                ("human", "{question}"),
            ]
        )
)

question, semantic_search, response = rag.query(question="Explain to me how to use kafka")

# Question

In [46]:
Markdown(
    question
)

Explain to me how to use kafka

In [47]:
Markdown(
     semantic_search
)

According to https://github.com/dpkp/kafka-python/
“DUE TO ISSUES WITH RELEASES, IT IS SUGGESTED TO USE https://github.com/wbarnha/kafka-python-ng FOR THE TIME BEING”
Use pip install kafka-python-ng instead
For example, when running JsonConsumer.java, got:
Consuming form kafka started
RESULTS:::0
RESULTS:::0
RESULTS:::0
Or when running JsonProducer.java, got:
Exception in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed
Solution:
Make sure in the scripts in src/main/java/org/example/ that you are running (e.g. JsonConsumer.java, JsonProducer.java), the StreamsConfig.BOOTSTRAP_SERVERS_CONFIG is the correct server url (e.g. europe-west3 from example vs europe-west2)
Make sure cluster key and secrets are updated in src/main/java/org/example/Secrets.java (KAFKA_CLUSTER_KEY and KAFKA_CLUSTER_SECRET)
If you have this error, it most likely that your kafka broker docker container is not working.
Use docker ps to confirm
Then in the docker compose yaml file folder, run docker compose up -d to start all the instances.
Use seed-kafka instead of stream-kafka to get a static set of results.
It is best to use the order by and limit clause on the query to the materialized view instead of the materialized view creation in order to guarantee consistent results
Homework - The answers in the homework do not match the provided options: You must follow the following steps: 1. clean-cluster 2. docker volume prune and use seed-kafka instead of stream-kafka. Ensure that the number of records is 100K.

In [49]:
Markdown(
    response.content
)

To use Kafka, you need to follow these steps:

1. **Installation**: 
   - It is recommended to use the `kafka-python-ng` package due to issues with releases in the original `kafka-python` package. You can install it using pip:
     ```
     pip install kafka-python-ng
     ```

2. **Configuration**:
   - Ensure that the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` in your scripts (e.g., `JsonConsumer.java`, `JsonProducer.java`) is set to the correct server URL.
   - Update the cluster key and secrets in your `Secrets.java` file (`KAFKA_CLUSTER_KEY` and `KAFKA_CLUSTER_SECRET`).

3. **Running Kafka**:
   - If you encounter issues such as authentication failures, check if your Kafka broker Docker container is running using `docker ps`.
   - If not running, navigate to the Docker compose YAML file folder and execute:
     ```
     docker compose up -d
     ```
   - This command will start all the necessary instances.

4. **Data Consistency**:
   - Use `seed-kafka` instead of `stream-kafka` to obtain a static set of results.
   - For consistent results, apply the `ORDER BY` and `LIMIT` clauses on the query to the materialized view rather than during the materialized view creation.

5. **Homework and Testing**:
   - If working on homework or testing, ensure to follow the steps: clean the cluster, prune Docker volumes, and use `seed-kafka` to ensure the number of records is 100K.

By following these steps, you should be able to set up and use Kafka effectively.